![image](https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use AutoAI RAG with predefined Milvus index to create a pattern about IBM

#### Disclaimers

- Use only Projects and Spaces that are available in watsonx context.


## Notebook content

This notebook contains the steps and code to demonstrate the usage of IBM AutoAI RAG with predefined vector store collection. Although this example uses Milvus, Elasticsearch and Chroma databases can be used similarly. Note that the AutoAI RAG experiment conducted in this notebook uses data scraped from the `ibm-watsonx-ai` SDK documentation.

Some familiarity with Python is helpful. This notebook uses Python 3.11.


## Learning goal

The learning goals of this notebook are:

- Create an AutoAI RAG job that will find the best RAG pattern based on collection created from `ibm-watsonx-ai` SDK documentation.


## Contents

This notebook contains the following parts:

- [Setup](#setup)
- [Index creation](#index)
- [RAG Optimizer definition](#definition)
- [RAG Experiment run](#run)
- [RAG Patterns comparison and testing](#comparison)
- [Historical runs](#runs)
- [Clean up](#cleanup)
- [Summary and next steps](#summary)

<a id="setup"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Create a <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance (a free plan is offered and information about how to create the instance can be found <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp" target="_blank" rel="noopener no referrer">here</a>).

### Install and import the required modules and dependencies

In [1]:
%pip install -U 'ibm-watsonx-ai[rag]>=1.4.6' | tail -n 1

Note: you may need to restart the kernel to use updated packages.


### Defining the watsonx.ai credentials
This cell defines the credentials required to work with the watsonx.ai Runtime service.

**Action:** Provide the IBM Cloud user API key. For details, see <a href="https://cloud.ibm.com/docs/account?topic=account-userapikey&interface=ui" target="_blank" rel="noopener no referrer">documentation</a>.

In [2]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=getpass.getpass("Please enter your watsonx.ai api key (hit enter): "),
)

### Working with spaces

You need to create a space that will be used for your work. If you do not have a space, you can use [Deployment Spaces Dashboard](https://dataplatform.cloud.ibm.com/ml-runtime/spaces?context=wx) to create one.

- Click **New Deployment Space**
- Create an empty space
- Select Cloud Object Storage
- Select watsonx.ai Runtime instance and press **Create**
- Go to **Manage** tab
- Copy `Space GUID` into your env file or else enter it in the window which will show up after running below cell

**Tip**: You can also use SDK to prepare the space for your work. More information can be found [here](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Space%20management.ipynb).

**Action**: assign space ID below

In [3]:
import os

try:
    space_id = os.environ["SPACE_ID"]
except KeyError:
    space_id = input("Please enter your space_id (hit enter): ")

Create an instance of APIClient with authentication details.

In [4]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials=credentials, space_id=space_id)

<a id="index"></a>

## Index creation

### Defining a connection to knowledge base

Provide id of connection to your knowledge database or create a new one. You can add connection on watsonx platform or type your credentials after running the code below.

In [5]:
vector_store_connection_id = (
    input(
        "Provide connection asset ID in your space. Skip this, if you wish to type credentials by hand and hit enter: "
    )
    or None
)

if vector_store_connection_id is None:
    try:
        username = os.environ["USERNAME"]
    except KeyError:
        username = input("Please enter your Milvus user name and hit enter: ")
    try:
        password = os.environ["PASSWORD"]
    except KeyError:
        password = getpass.getpass("Please enter your Milvus password and hit enter: ")
    try:
        host = os.environ["HOST"]
    except KeyError:
        host = input("Please enter your Milvus hostname and hit enter: ")
    try:
        port = os.environ["PORT"]
    except KeyError:
        port = input("Please enter your Milvus port number and hit enter: ")
    try:
        ssl = os.environ["SSL"]
    except:
        ssl = bool(
            input(
                "Please enter ('y'/anything) if your Milvus instance has SSL enabled. Skip if it is not: "
            )
        )

    # Create connection
    milvus_data_source_type_id = client.connections.get_datasource_type_uid_by_name(
        "milvus"
    )
    details = client.connections.create(
        {
            client.connections.ConfigurationMetaNames.NAME: "Milvus Connection - sample notebook",
            client.connections.ConfigurationMetaNames.DESCRIPTION: "Connection created by the sample notebook",
            client.connections.ConfigurationMetaNames.DATASOURCE_TYPE: milvus_data_source_type_id,
            client.connections.ConfigurationMetaNames.PROPERTIES: {
                "host": host,
                "port": port,
                "username": username,
                "password": password,
                "ssl": ssl,
            },
        }
    )

    vector_store_connection_id = client.connections.get_id(details)

Download example data. You can also assign your own text to `document content`.

In [6]:
import requests

url = "https://ibm.github.io/watsonx-ai-python-sdk/v1.3.42/base.html"

response = requests.get(url)
response.raise_for_status()

document_content = response.text

Chunk and upload your document to the vector store.

In [7]:
from ibm_watsonx_ai.foundation_models.embeddings import Embeddings
from ibm_watsonx_ai.foundation_models.extensions.rag.chunker import LangChainChunker
from ibm_watsonx_ai.foundation_models.extensions.rag.vector_stores import (
    MilvusVectorStore,
)
from langchain_core.documents import Document

# Defining vector store from the connection id
embedding = Embeddings(model_id="ibm/slate-125m-english-rtrvr-v2", api_client=client)
vector_store = MilvusVectorStore(
    api_client=client,
    connection_id=vector_store_connection_id,
    collection_name="collection_notebook_sample",
    embedding_function=embedding,
    drop_old=True,
)

# Chunking document into smaller segments
document = Document(
    page_content=document_content, metadata={"document_id": "base.html"}
)
text_splitter = LangChainChunker(method="recursive", chunk_size=256, chunk_overlap=32)
chunks = text_splitter.split_documents([document])

# Uploading document to vector store
ids = vector_store.add_documents(chunks, batch_size=300)

print(ids[:5])

['e1152397722fe38aed87100b0da9aca5e1780efffc8a3d5cfd635ddc5af59269', '4a9dd2ad8fd9b7328e4fc0492987d506bb0f08d0896c906892068b6f474b4797', '6bad206f416d5c7453c6c57bc3b24b0464d46c798f9d590c95c7fdc653909afc', '664773b7d870a46ed5974a600a0a1f15a9f4b62a40e2f8fc83e3753b396de1a7', '615e519eb7710ea9a58781d9c0d230700dd15e6ebda69c24f369f8132a6d756b']


<a id="definition"></a>

## RAG Optimizer definition

### Defining a connection to vector store

Define a reference to knowledge base.

In [8]:
from ibm_watsonx_ai.helpers import DataConnection
from ibm_watsonx_ai.utils.autoai.enums import KnowledgeBaseFieldRole
from ibm_watsonx_ai.utils.autoai.knowledge_base import VectorStoreKnowledgeBase

connection = DataConnection(connection_asset_id=vector_store_connection_id)
connection.set_client(client)

vector_store_knowledge_base_references = [
    VectorStoreKnowledgeBase(
        name="Embedded base.html file",
        description="This knowledge base contains samples from watsonx.ai sdk documentation.",
        connection=connection,
        settings={
            "index_name": "collection_notebook_sample",
            "fields_mapping": [
                {
                    "role": KnowledgeBaseFieldRole.DENSE_VECTOR_EMBEDDINGS,
                    "field_name": "vector",
                },
                {
                    "role": KnowledgeBaseFieldRole.DOCUMENT_NAME,
                    "field_name": "document_id",
                },
                {
                    "role": KnowledgeBaseFieldRole.TEXT,
                    "field_name": "text",
                },
                {
                    "role": KnowledgeBaseFieldRole.CHUNK_SEQUENCE_NUMBER,
                    "field_name": "sequence_number",
                },
            ],
            "embeddings": {"model_id": "ibm/slate-125m-english-rtrvr-v2"},
        },
    )
]

### Defining a connection to test data

Upload a `json` file that will be used for benchmarking to COS and then define a connection to this file. 
Define benchmarking question about your knowledge base. Replace the questions below.

In [9]:
benchmarking_data_IBM_page_content = [
    {
        "question": "How can you set or refresh user request headers using the APIClient class?",
        "correct_answer": "client.set_headers({'Authorization': 'Bearer <token>'})",
        "correct_answer_document_ids": ["base.html"],
    },
    {
        "question": "How to initialise Credentials object with api_key",
        "correct_answer": "credentials = Credentials(url = 'https://us-south.ml.cloud.ibm.com', api_key = '***********')",
        "correct_answer_document_ids": ["base.html"],
    },
]

Upload testing data to the bucket as a `json` file.

In [10]:
import json

test_filename = "benchmarking_data_predefined_vector_store_sample.json"

if not os.path.isfile(test_filename):
    with open(test_filename, "w") as json_file:
        json.dump(benchmarking_data_IBM_page_content, json_file, indent=4)

test_asset_details = client.data_assets.create(
    name=test_filename, file_path=test_filename
)

test_asset_id = client.data_assets.get_id(test_asset_details)
test_asset_id

Creating data asset...
SUCCESS


'7ef06995-b9ca-4c2f-9a8e-3e490301c9f5'

Define connection information to testing data.

In [11]:
test_data_references = [DataConnection(data_asset_id=test_asset_id)]

### RAG Optimizer configuration

Provide the input information for AutoAI RAG optimizer:
- `name` - experiment name
- `description` - experiment description
- `max_number_of_rag_patterns` - maximum number of RAG patterns to create
- `optimization_metrics` - target optimization metrics

In [12]:
from ibm_watsonx_ai.experiment import AutoAI
from ibm_watsonx_ai.foundation_models.schema import (
    AutoAIRAGGenerationConfig,
    AutoAIRAGModelConfig,
)

experiment = AutoAI(
    credentials=credentials,
    space_id=space_id,
)

foundation_model = AutoAIRAGModelConfig(
    model_id="mistralai/mistral-small-3-1-24b-instruct-2503",
)

generation_config = AutoAIRAGGenerationConfig(
    foundation_models=[foundation_model],
)

rag_optimizer = experiment.rag_optimizer(
    name="AutoAI RAG - sample notebook - knowledge base",
    description="Experiment run in sample notebook",
    generation=generation_config,
    max_number_of_rag_patterns=3,
    optimization_metrics=[AutoAI.RAGMetrics.ANSWER_CORRECTNESS],
)

Configuration parameters can be retrieved via `get_params()`.

In [13]:
rag_optimizer.get_params()

{'name': 'AutoAI RAG - sample notebook - knowledge base',
 'description': 'Experiment run in sample notebook',
 'max_number_of_rag_patterns': 3,
 'optimization_metrics': ['answer_correctness'],
 'generation': {'foundation_models': [{'model_id': 'mistralai/mistral-small-3-1-24b-instruct-2503'}]}}

<a id="run"></a>
## RAG Experiment run

Call the `run()` method to trigger the AutoAI RAG experiment. You can either use interactive mode (synchronous job) or background mode (asynchronous job) by specifying `background_mode=True`.

In [14]:
run_details = rag_optimizer.run(
    knowledge_base_references=vector_store_knowledge_base_references,
    test_data_references=test_data_references,
    background_mode=False,
)



##############################################

Running '7b817550-66ae-4c80-8b74-7aceac97c9d9'

##############################################


pending..............
running......
completed
Training of '7b817550-66ae-4c80-8b74-7aceac97c9d9' finished successfully.


You can use the `get_run_status()` method to monitor AutoAI RAG jobs in background mode.

In [15]:
rag_optimizer.get_run_status()

'completed'

<a id="comparison"></a>
## Comparison and testing of RAG Patterns

You can list the trained patterns and information on evaluation metrics in the form of a Pandas DataFrame by calling the `summary()` method. You can use the DataFrame to compare all discovered patterns and select the one you like for further testing.

In [16]:
summary = rag_optimizer.summary()
summary

,mean_answer_correctness,mean_faithfulness,mean_context_correctness,chunking.method,chunking.chunk_size,chunking.chunk_overlap,embeddings.model_id,vector_store.distance_metric,retrieval.method,retrieval.number_of_chunks,retrieval.hybrid_ranker,generation.model_id
Pattern_Name,,,,,,,,,,,,
Pattern1,0.7083,0.0867,1.0,,,,,,,,,mistralai/mistral-small-3-1-24b-instruct-2503
Pattern2,0.7083,0.1228,1.0,,,,,,,,,mistralai/mistral-small-3-1-24b-instruct-2503
Pattern3,0.5833,0.1396,1.0,,,,,,,,,mistralai/mistral-small-3-1-24b-instruct-2503


Additionally, you can pass the `scoring` parameter to the summary method, to filter RAG patterns starting with the best.

```python
summary = rag_optimizer.summary(scoring="faithfulness")
```

In [17]:
rag_optimizer.get_run_details()

{'entity': {'hardware_spec': {'id': 'a6c4923b-b8e4-444c-9f43-8a7ec3020110',
   'name': 'L'},
  'knowledge_base_references': [{'description': 'This knowledge base contains samples from watsonx.ai sdk documentation.',
    'name': 'Embedded base.html file',
    'reference': {'connection': {'id': 'b05e66e7-380c-47ff-a692-56702e005753'},
     'location': {},
     'type': 'connection_asset'},
    'settings': {'embeddings': {'model_id': 'ibm/slate-125m-english-rtrvr-v2'},
     'fields_mapping': [{'field_name': 'vector',
       'role': 'dense_vector_embeddings'},
      {'field_name': 'document_id', 'role': 'document_name'},
      {'field_name': 'text', 'role': 'text'},
      {'field_name': 'sequence_number', 'role': 'chunk_sequence_number'}],
     'index_name': 'collection_notebook_sample'},
    'type': 'vector_store'}],
  'parameters': {'constraints': {'generation': {'foundation_models': [{'model_id': 'mistralai/mistral-small-3-1-24b-instruct-2503'}]},
    'max_number_of_rag_patterns': 3},
  

### Get selected pattern

Get the RAGPattern object from the RAG Optimizer experiment. By default, the RAGPattern of the best pattern is returned.

In [18]:
best_pattern_name = summary.index.values[0]
print("Best pattern is:", best_pattern_name)

best_pattern = rag_optimizer.get_pattern()

Best pattern is: Pattern1


  Using cached pyarrow-22.0.0-cp311-cp311-macosx_12_0_arm64.whl.metadata (3.1 kB)
Using cached pyarrow-22.0.0-cp311-cp311-macosx_12_0_arm64.whl (34.3 MB)


The pattern details can be retrieved by calling the `get_pattern_details` method:

```python
rag_optimizer.get_pattern_details(pattern_name='Pattern2')
```

Query the RAGPattern locally, to test it.

In [19]:
from ibm_watsonx_ai.deployments import RuntimeContext

runtime_context = RuntimeContext(api_client=client)
inference_service_function = best_pattern.inference_service(runtime_context)[0]

In [20]:
question = "How to add Task Credentials?"

context = RuntimeContext(
    api_client=client,
    request_payload_json={"messages": [{"role": "user", "content": question}]},
)

inference_service_function(context)

{'body': {'choices': [{'index': 0,
    'message': {'role': 'system',
     'content': 'Based on the provided document titled "IBM watsonx.ai for IBM Cloud", here\'s how you can add task credentials:\n\n1. **Using the `Credentials` class:**\n\nYou can create credentials using an API key like this:\n\n```python\nfrom ibm_watsonx_ai import Credentials\n\ncredentials = Credentials(\n    url="https://us-south.ml.cloud.ibm.com",\n    api_key=IAM_API_KEY\n)\n```\n\nOr, you can create credentials from a dictionary:\n\n```python\nfrom ibm_watsonx_ai import Credentials\n\ncredentials = Credentials.from_dict({\n    \'url\': "<url>",\n    \'apikey\': IAM_API_KEY\n})\n```\n\n2. **Using the `APIClient` class:**\n\nYou can also add credentials when creating an `APIClient` instance:\n\n```python\nfrom ibm_watsonx_ai import APIClient, Credentials\n\ncredentials = Credentials(\n    url="<url>",\n    api_key=IAM_API_KEY\n)\n\nclient = APIClient(credentials, space_id=<space_id>)\n```\n\nIn these examples, 

### Deploy RAGPattern

Deployment is done by storing the defined RAG function and then by creating a deployed asset.

In [21]:
deployment_details = best_pattern.inference_service.deploy(
    name="AutoAI RAG deployment - ibm_watsonx_ai documentation",
    space_id=space_id,
    deploy_params={"tags": ["wx-autoai-rag"]},
)



######################################################################################

Synchronous deployment creation for id: '6ed548d3-6ced-482b-a512-591401ea6b96' started

######################################################################################


initializing
Note: online_url and serving_urls are deprecated and will be removed in a future release. Use inference instead.
.....
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='c2de3690-471a-4f4d-8ee2-e051f07bb5df'
-----------------------------------------------------------------------------------------------




### Test the deployed function

RAG service is now deployed in our space. To test our solution we can run the cell below. Questions have to be provided in the payload. Their format is provided below.

In [22]:
deployment_id = client.deployments.get_id(deployment_details)

payload = {"messages": [{"role": "user", "content": question}]}
score_response = client.deployments.run_ai_service(deployment_id, payload)

In [23]:
print(score_response["choices"][0]["message"]["content"])

Based on the provided document titled "IBM watsonx.ai for IBM Cloud", here's how you can add task credentials:

1. **Using the `Credentials` class directly:**

```python
from ibm_watsonx_ai import Credentials

credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=IAM_API_KEY
)
```

2. **Using the `from_dict` method:**

```python
from ibm_watsonx_ai import Credentials

credentials = Credentials.from_dict({
    'url': '<url>',
    'apikey': IAM_API_KEY
})
```

In both examples, replace `<url>` with the appropriate URL and `IAM_API_KEY` with your actual API key.

3. **Setting credentials in an `APIClient`:**

```python
from ibm_watsonx_ai import APIClient, Credentials

credentials = Credentials(
    url="<url>",
    api_key=IAM_API_KEY
)

client = APIClient(credentials, space_id=<space_id>)
```

Replace `<url>` with the appropriate URL, `IAM_API_KEY` with your actual API key, and `<space_id>` with your space ID.

These examples demonstrate how to create and s

<a id="runs"></a>
## Historical runs

In this section you learn to work with historical RAG Optimizer jobs (runs).

To list historical runs use the `list()` method and provide the `'rag_optimizer'` filter.

In [24]:
experiment.runs(filter="rag_optimizer").list()

In [25]:
run_id = run_details["metadata"]["id"]
run_id

'7b817550-66ae-4c80-8b74-7aceac97c9d9'

### Get executed optimizer's configuration parameters

In [26]:
experiment.runs.get_rag_params(run_id=run_id)

{'name': 'AutoAI RAG - sample notebook - knowledge base',
 'description': 'Experiment run in sample notebook',
 'max_number_of_rag_patterns': 3,
 'generation': {'foundation_models': [{'model_id': 'mistralai/mistral-small-3-1-24b-instruct-2503'}]},
 'optimization_metrics': ['answer_correctness']}

### Get historical rag_optimizer instance and training details

In [27]:
historical_opt = experiment.runs.get_rag_optimizer(run_id)

### List trained patterns for selected optimizer

In [28]:
historical_opt.summary()

,mean_answer_correctness,mean_faithfulness,mean_context_correctness,chunking.method,chunking.chunk_size,chunking.chunk_overlap,embeddings.model_id,vector_store.distance_metric,retrieval.method,retrieval.number_of_chunks,retrieval.hybrid_ranker,generation.model_id
Pattern_Name,,,,,,,,,,,,
Pattern1,0.7083,0.0867,1.0,,,,,,,,,mistralai/mistral-small-3-1-24b-instruct-2503
Pattern2,0.7083,0.1228,1.0,,,,,,,,,mistralai/mistral-small-3-1-24b-instruct-2503
Pattern3,0.5833,0.1396,1.0,,,,,,,,,mistralai/mistral-small-3-1-24b-instruct-2503


<a id="cleanup"></a>
## Clean up

To delete the current experiment, use the `cancel_run` method.

**Warning:** Be careful: once you delete an experiment, you will no longer be able to refer to it.

In [29]:
rag_optimizer.cancel_run(hard_delete=True)

'SUCCESS'

To delete the deployment, use the `delete` method. 

**Warning:** Keeping the deployment active may lead to unnecessary consumption of Compute Unit Hours (CUHs).

In [30]:
client.deployments.delete(deployment_id)

'SUCCESS'

To delete obsolete collections, us the `clear` method of `MilvusVectorStore`

In [31]:
vector_store.clear()

If you want to clean up all created assets:
- experiments
- trainings
- pipelines
- model definitions
- models
- functions
- deployments

please follow up this sample [notebook](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Machine%20Learning%20artifacts%20management.ipynb).

<a id="summary"></a>
## Summary and next steps

You successfully completed this notebook!

You learned how to use `ibm-watsonx-ai` to run AutoAI RAG experiments. 

 Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors

**Paweł Kocur**, Software Engineer watsonx.ai

Copyright © 2025 IBM. This notebook and its source code are released under the terms of the MIT License.